## 연습 1: TelConnect Unity Catalog 구조 설정

이 연습에서는 Lakeflow 작업 태스크가 읽고 쓸 Unity Catalog 기반을 준비합니다. 메달리온 아키텍처 패턴에 따라 세 개의 스키마를 가진 카탈로그를 만듭니다.

- `bronze` — 수집된 원시 CDR 레코드
- `silver` — 정제 및 검증된 CDR 데이터
- `gold` — 보고용 집계 네트워크 사용량 지표

### ✅ 제공됨: 처리 날짜에 대한 작업 매개변수 수신

이 노트북이 Lakeflow 작업 태스크로 실행되면 작업은 `processing_date` 매개변수를 자동으로 전달합니다. `dbutils.widgets` API를 통해 노트북 내부에서 해당 값을 사용할 수 있습니다.

아래 셀을 실행하여 위젯을 등록하세요. 수동으로 실행하는 경우 기본값은 오늘 날짜입니다.

In [ ]:
# ✅ 이 셀을 실행하세요 — 작업 매개변수 위젯을 등록합니다

import datetime

dbutils.widgets.text("processing_date", str(datetime.date.today()), "Processing Date (YYYY-MM-DD)")
processing_date = dbutils.widgets.get("processing_date")

print(f"처리 날짜: {processing_date}")

### ✅ 제공됨: Unity Catalog 구조 만들기

아래 셀을 실행하여 `telconnect_lab` 카탈로그와 세 개의 메달리온 스키마를 만드세요. 이 구조는 랩 전반에 걸쳐 사용됩니다.


In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로 기본 카탈로그 루트의 스토리지 경로를 상속합니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"스토리지 루트: {storage_root}")

In [ ]:
# ✅ 정답

catalog = "telconnect_lab"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog} MANAGED LOCATION '{storage_root}' COMMENT 'TelConnect 네트워크 분석 플랫폼'")
print(f"✅ 카탈로그 '{catalog}'이(가) 생성되었습니다.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze COMMENT '수집된 원시 통화 상세 기록(CDR)'")
print(f"✅ 스키마 '{catalog}.bronze'이(가) 생성되었습니다.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.silver COMMENT '정제 및 검증된 CDR 데이터'")
print(f"✅ 스키마 '{catalog}.silver'이(가) 생성되었습니다.")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold COMMENT '보고용 집계 네트워크 사용량 지표'")
print(f"✅ 스키마 '{catalog}.gold'이(가) 생성되었습니다.")

## 연습 2: 원시 CDR 데이터 수집 (Bronze 계층)

Bronze 계층은 수신된 그대로의 원시 통화 상세 기록(CDR)을 저장합니다. 단일 처리일의 네트워크 이벤트를 나타내는 샘플 CSV가 제공됩니다. 이 파일을 작업 영역에 로드하고 Bronze Delta 테이블로 등록한 뒤 레코드 수를 확인합니다.

### ✅ 제공됨: raw_uploads 볼륨 만들기 및 샘플 데이터 다운로드

아래 셀을 실행하여 Unity Catalog 볼륨을 만들고 GitHub에서 `telecom_cdrs.csv`를 볼륨으로 자동 다운로드하세요.


In [ ]:
# ✅ 이 셀을 실행하세요 — raw_uploads 볼륨을 만들고 샘플 CDR 데이터를 다운로드합니다

import urllib.request

spark.sql("CREATE VOLUME IF NOT EXISTS telconnect_lab.bronze.raw_uploads COMMENT '원시 CDR 파일을 위한 랜딩 영역'")
print("✅ 볼륨 telconnect_lab.bronze.raw_uploads이(가) 준비되었습니다.")

url = "https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Allfiles/data/telecom_cdrs.csv"
dest = "/Volumes/telconnect_lab/bronze/raw_uploads/telecom_cdrs.csv"
urllib.request.urlretrieve(url, dest)
print(f"✅ telecom_cdrs.csv 파일을 {dest}에 다운로드했습니다.")


### ✅ 제공됨: CDR CSV를 DataFrame으로 로드

아래 셀을 실행하여 업로드된 CSV를 Spark DataFrame으로 읽어오세요. 출력을 살펴보고, 연습 3에서 정제할 품질 문제가 있는 레코드를 확인하세요.


In [ ]:
# ✅ 이 셀을 실행하세요 — 볼륨에서 원시 CDR CSV를 로드합니다

from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

cdr_schema = StructType([
    StructField("call_id",           StringType(),  True),
    StructField("caller_msisdn",     StringType(),  True),
    StructField("callee_msisdn",     StringType(),  True),
    StructField("call_start_time",   StringType(),  True),
    StructField("call_end_time",     StringType(),  True),
    StructField("duration_seconds",  IntegerType(), True),
    StructField("call_type",         StringType(),  True),
    StructField("call_status",       StringType(),  True),
    StructField("network_type",      StringType(),  True),
    StructField("cell_tower_id",     StringType(),  True),
    StructField("region",            StringType(),  True),
    StructField("roaming",           StringType(),  True),
    StructField("data_mb",           DoubleType(),  True),
    StructField("sms_count",         IntegerType(), True),
])

raw_df = (
    spark.read
    .option("header", True)
    .schema(cdr_schema)
    .csv("/Volumes/telconnect_lab/bronze/raw_uploads/telecom_cdrs.csv")
)

raw_df.display()
print(f"\n로드된 원시 CDR 레코드 수: {raw_df.count()}")


### ✅ 제공됨: 원시 CDR 데이터를 Bronze Delta 테이블에 쓰기

아래 셀을 실행하여 `raw_df`를 Unity Catalog의 관리형 Delta 테이블로 저장하세요. 이는 모든 다운스트림 정제 및 집계의 출발점입니다.


In [ ]:
# ✅ 정답

raw_df.write.format("delta").mode("overwrite").saveAsTable("telconnect_lab.bronze.cdr_bronze")

count = spark.table("telconnect_lab.bronze.cdr_bronze").count()
print(f"✅ Bronze 테이블이 기록되었습니다. 레코드 수: {count}")

## 연습 3: CDR 데이터 정제 및 검증 (Silver 계층)

Silver 계층에는 검증을 통과한 CDR이 포함됩니다. 운영 파이프라인에서는 정제된 레코드만 다운스트림의 집계 및 청구 시스템으로 전달됩니다.

원시 데이터를 주의 깊게 검토하세요. 다음과 같이 수정해야 할 품질 문제를 발견하게 됩니다.

| Issue | Example | Handling rule |
|---|---|---|
| Negative duration | CDR010 | Drop records where `duration_seconds < 0` |
| Missing callee for VOICE/DATA calls | CDR007 | Drop VOICE and DATA records where `callee_msisdn` is null |
| Zero-duration completed calls | Several | Drop VOICE records with `call_status = COMPLETED` and `duration_seconds = 0` |
| Invalid call_type value | Check data | Keep only: `VOICE`, `DATA`, `SMS` |

In [ ]:
# ✅ 이 셀을 실행하세요 — 연습 2에서 만든 bronze 테이블에서 읽어옵니다

bronze_df = spark.table("telconnect_lab.bronze.cdr_bronze")
print(f"Bronze 레코드 수: {bronze_df.count()}")
bronze_df.display()

### ✅ Provided: Apply cleaning rules and write to Silver

Run the cell below. It applies the four data quality rules to `bronze_df` and writes the validated records to the Silver table. Review the printout to see how many records were dropped at each step.


In [ ]:
# ✅ 정답

from pyspark.sql.functions import col

bronze_count = bronze_df.count()

silver_df = (
    bronze_df
    # 규칙 1: 음수 통화 시간 제거
    .filter(col("duration_seconds") >= 0)
    # 규칙 2: 수신자가 없는 VOICE 및 DATA 레코드 제거
    .filter(
        ~((col("call_type").isin("VOICE", "DATA")) & col("callee_msisdn").isNull())
    )
    # 규칙 3: 통화 시간이 0인 완료된 VOICE 통화 제거
    .filter(
        ~((col("call_type") == "VOICE") & (col("call_status") == "COMPLETED") & (col("duration_seconds") == 0))
    )
    # 규칙 4: 유효한 통화 유형만 유지
    .filter(col("call_type").isin("VOICE", "DATA", "SMS"))
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("telconnect_lab.silver.cdr_silver")

silver_count = spark.table("telconnect_lab.silver.cdr_silver").count()
print(f"✅ Silver 테이블이 기록되었습니다.")
print(f"   Bronze 레코드 : {bronze_count}")
print(f"   Silver 레코드 : {silver_count}")
print(f"   제거된 레코드 : {bronze_count - silver_count}")

## 연습 4: 네트워크 지표 집계 (Gold 계층)

Gold 계층에는 대시보드와 운영 팀이 사용하는 집계 지표가 포함됩니다. TelConnect의 네트워크 운영 센터는 지역 및 네트워크 유형별로 다음 항목을 보여주는 일일 요약이 필요합니다.

- 처리된 총 통화 수
- 성공한 총 통화 수(`COMPLETED` 상태)
- 통화 드롭률(`DROPPED`된 통화의 비율)
- 평균 통화 시간(`VOICE` 통화만)
- 전송된 총 데이터(MB)(`DATA` 통화만)

In [ ]:
# ✅ 이 셀을 실행하세요 — silver 테이블에서 읽어옵니다

silver_df = spark.table("telconnect_lab.silver.cdr_silver")
print(f"Silver 레코드 수: {silver_df.count()}")

### ✅ 제공됨: 지역별 네트워크 요약 작성

아래 셀을 실행하세요. Silver CDR 데이터를 지역 및 네트워크 유형별로 집계하여 다섯 가지 운영 지표를 계산하고 그 결과를 Gold 테이블에 씁니다.


In [ ]:
# ✅ 정답

from pyspark.sql.functions import (
    count, sum as spark_sum, avg, when, round as spark_round
)

gold_df = (
    silver_df
    .groupBy("region", "network_type")
    .agg(
        count("*").alias("total_calls"),
        count(when(col("call_status") == "COMPLETED", 1)).alias("completed_calls"),
        spark_round(
            count(when(col("call_status") == "DROPPED", 1)) * 100.0 / count("*"), 2
        ).alias("drop_rate_pct"),
        spark_round(
            avg(when((col("call_type") == "VOICE") & (col("duration_seconds") > 0), col("duration_seconds"))), 1
        ).alias("avg_voice_duration_sec"),
        spark_round(
            spark_sum(when(col("call_type") == "DATA", col("data_mb"))), 2
        ).alias("total_data_mb"),
    )
    .orderBy("region", "network_type")
)

gold_df.write.format("delta").mode("overwrite").saveAsTable("telconnect_lab.gold.network_summary")

print(f"✅ Gold 테이블이 기록되었습니다. 행 수: {spark.table('telconnect_lab.gold.network_summary').count()}")
spark.table("telconnect_lab.gold.network_summary").display()

## 연습 5: 노트북을 작업 실행에 적합하게 만들기

이 노트북이 Lakeflow 작업 태스크로 실행되면 결과를 작업에 다시 전달해야 합니다. Databricks는 두 가지 메커니즘을 제공합니다.

- **`dbutils.notebook.exit()`** — 노트북을 종료하고 `Run Notebook` 태스크 유형을 통해 이 노트북을 호출하는 모든 태스크에 문자열 값을 전달합니다.
- **`dbutils.jobs.taskValues.set()`** — 동일한 작업 실행의 다운스트림 태스크가 검색할 수 있도록 명명된 값을 저장합니다.

여기서는 노트북을 관찰 가능하고 연결 가능하게 만들기 위해 두 가지를 모두 사용합니다.

### ✅ 제공됨: Gold 레코드 수 계산 및 결과를 작업과 공유

아래 셀을 실행하세요. Gold 행 수를 태스크 값으로 게시하여(다운스트림 작업 태스크가 읽을 수 있도록) 노트북을 구조화된 JSON 상태 문자열과 함께 종료합니다. 이것이 노트북을 적절한 Lakeflow 작업 태스크로 동작하게 만듭니다.


In [ ]:
# ✅ 정답

import json

gold_row_count = spark.table("telconnect_lab.gold.network_summary").count()

dbutils.jobs.taskValues.set(key="gold_row_count", value=gold_row_count)

print(f"✅ processing_date={processing_date}에 대한 파이프라인이 완료되었습니다. Gold 행 수: {gold_row_count}")

result = json.dumps({"status": "success", "processing_date": processing_date, "gold_rows": gold_row_count})
dbutils.notebook.exit(result)